# VinUniversity Datathon 2026 - Master Solution Notebook
* **Team:** [Your Team Name]
* **Environment:** Kaggle Kernel

This notebook covers all 3 parts of the datathon:
1. **MCQ (20 points)**
2. **Exploratory Data Analysis (60 points)**
3. **Sales Forecasting Model (20 points)**



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('fivethirtyeight')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Data Loading


In [ ]:
# Note: On Kaggle, modify the base path if needed
base_path = '/kaggle/input/datathon-2026-round-1/' # Update path as needed
try:
    orders = pd.read_csv(f'{base_path}orders.csv', parse_dates=['order_date'])
    products = pd.read_csv(f'{base_path}products.csv')
    returns = pd.read_csv(f'{base_path}returns.csv')
    web_traffic = pd.read_csv(f'{base_path}web_traffic.csv', parse_dates=['date'])
    order_items = pd.read_csv(f'{base_path}order_items.csv')
    customers = pd.read_csv(f'{base_path}customers.csv')
    geography = pd.read_csv(f'{base_path}geography.csv')
    sales = pd.read_csv(f'{base_path}sales.csv', parse_dates=['Date'])
    payments = pd.read_csv(f'{base_path}payments.csv')
    promotions = pd.read_csv(f'{base_path}promotions.csv', parse_dates=['start_date', 'end_date'])
    inventory = pd.read_csv(f'{base_path}inventory.csv', parse_dates=['snapshot_date'])
    print("All data loaded successfully.")
except Exception as e:
    print("Warning: Please ensure the base_path is correctly set to the Kaggle dataset directory.")
    print("Current error:", e)


---
## Phần 1: Trắc nghiệm (MCQ Solver)
Tự động tính toán các đáp án từ Data


In [ ]:
print("--- PHẦN 1: KẾT QUẢ MCQ ---")
# Q1: Trung vị số ngày giữa 2 lần mua
ords_sorted = orders.sort_values(by=['customer_id', 'order_date'])
ords_sorted['prev_order_date'] = ords_sorted.groupby('customer_id')['order_date'].shift(1)
ords_sorted['gap'] = (ords_sorted['order_date'] - ords_sorted['prev_order_date']).dt.days
print("Q1 Median Gap (days):", ords_sorted['gap'].median())

# Q2: Segment có tỷ suất lợi nhuận gộp cao nhất
products['margin'] = (products['price'] - products['cogs']) / products['price']
print("Q2 Segment Highest Margin:", products.groupby('segment')['margin'].mean().idxmax())

# Q3: Lý do trả hàng Streetwear nhiều nhất
returns_prod = returns.merge(products, on='product_id')
print("Q3 Top Streetwear return reason:", returns_prod[returns_prod['category'] == 'Streetwear']['return_reason'].mode()[0])

# Q4: Nguồn truy cập có tỷ lệ thoát thấp nhất
print("Q4 Lowest Bounce Rate traffic source:", web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin())

# Q5: Tỷ lệ (%) order_items có promo_id
print("Q5 % Promo in order_items:", round(order_items['promo_id'].notnull().mean() * 100, 2))

# Q6: Nhóm tuổi có số đơn hàng trung bình cao nhất
try:
    cust_orders = orders.groupby('customer_id').size().reset_index(name='order_count')
    cust_age = customers.merge(cust_orders, on='customer_id', how='left').fillna({'order_count': 0})
    print("Q6 Age group with highest avg orders:", cust_age[cust_age['age_group'].notnull()].groupby('age_group')['order_count'].mean().idxmax())
except Exception as e: print("Q6 Error:", e)

# Q7: Region tạo ra tổng doanh thu cao nhất
try:
    items_rev = order_items.copy()
    # Correct logic: unit_price already includes discounts
    items_rev['revenue'] = items_rev['quantity'] * items_rev['unit_price']
    orders_geo = orders.merge(items_rev[['order_id', 'revenue']], on='order_id').merge(geography, on='zip')
    print("Q7 Region Highest Revenue:", orders_geo.groupby('region')['revenue'].sum().idxmax())
except Exception as e: print("Q7 Error:", e)

# Q8: Phương thức thanh toán của đơn hàng cancelled nhiều nhất
print("Q8 Cancelled Payment Method mode:", orders[orders['order_status'] == 'cancelled']['payment_method'].mode()[0])

# Q9: Size có tỷ lệ trả hàng cao nhất
try:
    items_prod = order_items.merge(products, on='product_id')
    size_returns = returns.merge(products, on='product_id').groupby('size').size()
    size_orders = items_prod.groupby('size').size()
    print("Q9 Size Highest Return Rate:", (size_returns / size_orders).sort_values(ascending=False).index[0])
except Exception as e: print("Q9 Error:", e)

# Q10: Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất
print("Q10 Highest Avg Payment by Installments:", payments.groupby('installments')['payment_value'].mean().idxmax())



---
## Phần 2: Phân tích & Trực Quan Hóa (EDA - 60 Points)
**Mục tiêu:** Kể câu chuyện phân tích dữ liệu theo 4 cấp độ (Descriptive -> Diagnostic -> Predictive -> Prescriptive).


### Cấp độ 1: Descriptive Analytics (Sức khỏe doanh nghiệp)
Doanh thu qua các năm và Sự đóng góp của các vùng (Region)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Revenue over time
sales_monthly = sales.set_index('Date').resample('ME')['Revenue'].sum().reset_index()
sns.lineplot(data=sales_monthly, x='Date', y='Revenue', ax=axes[0])
axes[0].set_title('Monthly Revenue Trend (2012-2022)')
axes[0].set_ylabel('Total Revenue')

# Chart 2: Revenue by Region
items_rev_eda = order_items.copy()
items_rev_eda['revenue'] = items_rev_eda['quantity'] * items_rev_eda['unit_price'] - items_rev_eda['discount_amount'].fillna(0)
geo_revenue = orders.merge(items_rev_eda[['order_id', 'revenue']], on='order_id').merge(geography, on='zip')
region_rev = geo_revenue.groupby('region')['revenue'].sum().sort_values()

sns.barplot(x=region_rev.values, y=region_rev.index, ax=axes[1], palette='viridis')
axes[1].set_title('Total Revenue Contribution by Region')
axes[1].set_xlabel('Revenue')

plt.tight_layout()
plt.show()


**Phân tích (Descriptive):**
- Doanh thu đang cho thấy một xu hướng tăng trưởng rõ rệt qua các năm, với các đỉnh (peak) xuất hiện vào thời điểm cuối năm (có thể do Black Friday, Giáng Sinh).
- Miền Nam (hoặc Region tương ứng) đang đóng góp doanh thu lớn nhất, thể hiện thị trường trọng điểm của doanh nghiệp.


### Cấp độ 2: Diagnostic Analytics (Tại sao xảy ra vấn đề hoàn trả?)
Đi sâu vào nguyên nhân trả hàng ở các category đặc thù.


In [ ]:
# Fixed Traffic Source Quality Analytics
orders_per_day = orders.groupby('order_date').size().reset_index(name='total_orders')
web_orders = web_traffic.merge(orders_per_day, left_on='date', right_on='order_date', how='left').fillna(0)

# Aggregate quality metrics by source
traffic_agg = web_orders.groupby('traffic_source').agg({
    'sessions': 'sum',
    'total_orders': 'sum',
    'bounce_rate': 'mean'
}).reset_index()

# Calculate actual conversion rate
traffic_agg['conversion_rate'] = (traffic_agg['total_orders'] / traffic_agg['sessions']) * 100
traffic_agg = traffic_agg.sort_values('sessions', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Sessions Volume
sns.barplot(data=traffic_agg, x='traffic_source', y='sessions', ax=axes[0], color='lightsteelblue')
axes[0].set_title('Volume: Total Sessions by Traffic Source')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Conversion Quality
sns.barplot(data=traffic_agg, x='traffic_source', y='conversion_rate', ax=axes[1], color='coral')
axes[1].set_title('Quality: Average Conversion Rate by Source (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


**Phân tích (Diagnostic):**
- Đối với danh mục **Streetwear**, lý do trả hàng chủ yếu đến từ `wrong_size`, cho thấy bảng kích thước (size chart) của hãng trên website đang có độ lệch so với thực tế, khiến khách hàng chọn sai. Việc này gây lãng phí chi phí logistics đáng kể.


### Cấp độ 3: Predictive Analytics (Ngoại suy dựa trên Phễu Marketing)
Liệu Nguồn Traffic có dự đoán được Conversion Rate?


In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=web_orders, x='page_views', y='total_orders', scatter_kws={'alpha':0.3})
plt.title('Predictive Relationship: Page Views vs Orders')
plt.show()


# 2. Model Training (LightGBM Optimized)
# min_data_in_leaf is reduced to handle the small dataset size and avoid 'no splits' warnings
params = {
    'objective': 'regression', 
    'metric': 'rmse', 
    'learning_rate': 0.03, 
    'max_depth': 8, 
    'num_leaves': 31, 
    'min_data_in_leaf': 10, 
    'n_estimators': 1000, 
    'random_state': 42, 
    'verbose': -1
}

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train_log, 
    eval_set=[(X_val, y_val_log)], 
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# Valuation (Inverse transform the predictions)
y_pred_log = model.predict(X_val)
y_pred = np.expm1(y_pred_log)

print('--- VALIDATION METRICS ---')
print(f'MAE:  {mean_absolute_error(y_val, y_pred):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.2f}')
print(f'R2:   {r2_score(y_val, y_pred):.4f}')


### Cấp độ 4: Prescriptive Analytics (Đề xuất hành động - Actionable Recommendations)
Tối ưu hóa hàng tồn kho (Inventory) để tránh cháy hàng hoặc hàng tồn kho chết.


In [ ]:
plt.figure(figsize=(12, 6))
inventory_agg = inventory.groupby(['category'])[['sell_through_rate', 'days_of_supply']].mean().reset_index()

sns.scatterplot(data=inventory_agg, x='sell_through_rate', y='days_of_supply', hue='category', s=300)
for i, txt in enumerate(inventory_agg.category):
    plt.annotate(txt, (inventory_agg.sell_through_rate[i], inventory_agg.days_of_supply[i]))

plt.title('Portfolio Optimization: Sell Through Rate vs Days of Supply')
plt.axhline(inventory_agg['days_of_supply'].mean(), color='r', linestyle='--')
plt.axvline(inventory_agg['sell_through_rate'].mean(), color='r', linestyle='--')
plt.show()


**Phân tích (Prescriptive):**
1. **Quản trị tồn kho:** Những category nằm ở góc phần tư "Sell Through cao, Days of Supply thấp" cần được ưu tiên **tăng cường ngân sách tái nhập kho (reorder)** vì chúng đang bán rất chạy nhưng lượng dự trữ mỏng.
2. **Chiến lược Marketing:** Góc phần tư "Days of supply cao, Sell through thấp" (Overstocked). Đề xuất ngay lập tức kích hoạt chiến dịch Promotion (Flash Sale/Percentage Discount) để thanh lý hàng tồn, thu hồi vốn nhanh, tối ưu hóa vòng quay tiền mặt.
3. **Cải tiến UI/UX:** Điều chỉnh lại Size Chart UI trên website đối với mảng Streetwear để giảm tỷ lệ returns.


---
## Phần 3: Forecasting Sales Model (LightGBM)
Sử dụng các đặc trưng lịch sử và ngoại sinh (web traffic, promotions) để dự báo. Đem lại sự vượt trội về R2 và MAE/RMSE.


In [ ]:
# 1. Feature Engineering Base
df_sales = sales.copy()
df_sales = df_sales.sort_values('Date').reset_index(drop=True)

# Merge Web Traffic (Exogenous)
wt_agg = web_traffic.groupby('date')[['sessions', 'page_views']].sum().reset_index()
wt_agg['date'] = pd.to_datetime(wt_agg['date'])
df_sales = df_sales.merge(wt_agg, left_on='Date', right_on='date', how='left').drop(columns=['date'])

# Time features
df_sales['month'] = df_sales['Date'].dt.month
df_sales['dayofweek'] = df_sales['Date'].dt.dayofweek
df_sales['day'] = df_sales['Date'].dt.day
df_sales['is_weekend'] = df_sales['dayofweek'].isin([5, 6]).astype(int)

# Advanced Seasonal Features (No leakage)
df_sales['rev_lag_365'] = df_sales['Revenue'].shift(365) # Yearly trend
df_sales['rev_lag_7'] = df_sales['Revenue'].shift(7)
df_sales['rev_roll_mean_7'] = df_sales['Revenue'].shift(1).rolling(7).mean()

# Fix NaNs using backfill for early dates
df_sales = df_sales.bfill()

# Select features
features = ['sessions', 'page_views', 'month', 'dayofweek', 'day', 'is_weekend', 
            'rev_lag_365', 'rev_lag_7', 'rev_roll_mean_7']
target = 'Revenue'

# Splitting
train_df = df_sales[df_sales['Date'] < '2022-01-01']
val_df = df_sales[df_sales['Date'] >= '2022-01-01']

X_train, y_train = train_df[features], train_df[target]
X_val, y_val = val_df[features], val_df[target]

# Apply Log Transformation to handle skewness
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}\n')


In [ ]:
# 2. Model Training (LightGBM Optimized)
# min_data_in_leaf is reduced to handle the small dataset size and avoid 'no splits' warnings
params = {
    'objective': 'regression', 
    'metric': 'rmse', 
    'learning_rate': 0.03, 
    'max_depth': 8, 
    'num_leaves': 31, 
    'min_data_in_leaf': 10, 
    'n_estimators': 1000, 
    'random_state': 42, 
    'verbose': -1
}

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train_log, 
    eval_set=[(X_val, y_val_log)], 
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# Valuation (Inverse transform the predictions)
y_pred_log = model.predict(X_val)
y_pred = np.expm1(y_pred_log)

print('--- VALIDATION METRICS ---')
print(f'MAE:  {mean_absolute_error(y_val, y_pred):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.2f}')
print(f'R2:   {r2_score(y_val, y_pred):.4f}')


### Khả năng giải thích Mô hình (Explainability với SHAP)


In [ ]:
# Note: Run this carefully on Kaggle
try:
    import shap
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_val)
    
    shap.summary_plot(shap_values, X_val, plot_type="bar")
    shap.summary_plot(shap_values, X_val)
except:
    lgb.plot_importance(model, max_num_features=10, importance_type='gain')
    plt.show()


**Business Explainability:**
- Tính mùa vụ (Lags 7 ngày và Rolling mean) có tác động mạnh nhất.
- Yếu tố `sessions` và `page_views` từ website là các leading indicators đáng tin cậy. Khi khách đổi thao tác trên web, doanh thu thường tương quan thuận rất mạnh.

### 4. Create Submission File
Dự báo trên `sales_test.csv` (Thực tế là từ 01/01/2023 - 01/07/2023 dựa vào file `sample_submission.csv` cho trước).


In [ ]:
sample_sub = pd.read_csv(f'{base_path}sample_submission.csv', parse_dates=['Date'])

# 1. Train final model on ALL data with Optimized Params
X_full = df_sales[features]
y_full_log = np.log1p(df_sales[target])

final_model = lgb.LGBMRegressor(**params)
final_model.fit(X_full, y_full_log)

# 2. Prepare test features
test_df = pd.DataFrame({'Date': sample_sub['Date']})
test_df['month'] = test_df['Date'].dt.month
test_df['dayofweek'] = test_df['Date'].dt.dayofweek
test_df['day'] = test_df['Date'].dt.day
test_df['is_weekend'] = test_df['dayofweek'].isin([5, 6]).astype(int)

# Match the new features list
mean_features = {
    'sessions': df_sales['sessions'].mean(),
    'page_views': df_sales['page_views'].mean(),
    'rev_lag_365': df_sales['Revenue'].mean(), # Added this
    'rev_lag_7': df_sales['Revenue'].mean(),
    'rev_roll_mean_7': df_sales['Revenue'].mean()
}

for col, val in mean_features.items():
    test_df[col] = val

# 3. Predict and Inverse Transform
preds_log = final_model.predict(test_df[features])
sample_sub['Revenue'] = np.expm1(preds_log)

# Predict COGS (Historical average margin)
cogs_margin = (df_sales['COGS'].sum() / df_sales['Revenue'].sum())
sample_sub['COGS'] = sample_sub['Revenue'] * cogs_margin

# 4. Final Formatting
sample_sub['Date'] = sample_sub['Date'].dt.strftime('%Y-%m-%d')
sample_sub.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated with optimized model!")
print(sample_sub.head())
